# NLP Applications: Text Classification and Machine Translation

**Session:** 2026-11-09 · Text Project: ChatGPT-based Application  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking  

---

## 1. Sentiment Classification & Multi-Label Aspect Extraction

Text classification assigns predefined categories to an unstructured natural language sequence $x$:

$$\hat{y} = \arg\max_{c \in \mathcal{C}} P(y = c \mid x)$$

In **Aspect-Based Sentiment Analysis (ABSA)** and **Multi-Label Classification**, an input text may simultaneously belong to multiple non-mutually exclusive categories:

$$\hat{\mathbf{y}} = [y_1, y_2, \dots, y_K] \in \{0, 1\}^K$$

Common domain aspects in dining and hospitality reviews include:
- `service`: Waitstaff responsiveness, courtesy, order correctness.
- `food`: Taste, temperature, freshness, portion size.
- `price`: Affordability, perceived value, billing transparency.
- `ambience`: Decor, noise level, lighting, seating comfort.
- `miscellaneous`: Transit accessibility, parking, facilities.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

DATA_DIR = Path("../data")
with (DATA_DIR / "sentiment_classification_benchmarks.json").open(encoding="utf-8") as f:
    sentiment_data = json.load(f)

with (DATA_DIR / "multilabel_aspect_samples.json").open(encoding="utf-8") as f:
    aspect_data = json.load(f)

print(f"Loaded {len(sentiment_data)} sentiment cases and {len(aspect_data)} aspect cases.")


def build_sentiment_prompt(text: str, explain: bool = False) -> str:
    prompt = "Classify the following text into neutral, negative, positive."
    if explain:
        prompt += " Provide an explanation for why this label was chosen."
    prompt += f"\n\nText: {text}\nLabel:"
    return prompt


# Evaluate prompt formatting
p1 = build_sentiment_prompt("I think the food was okay.", explain=True)
print("=== SENTIMENT PROMPT ===")
print(p1)

## 2. Multi-Label Aspect Classification Engine

Below, we implement an aspect extraction engine that maps review statements to the target aspect categories (`service`, `food`, `price`, `ambience`, `miscellaneous`).


In [ ]:
def evaluate_aspects(text: str, target_aspects: list[str]) -> dict[str, bool]:
    """Rule-based aspect identifier matching target ontology for benchmark evaluation."""
    text_lower = text.lower()
    detected = {}

    aspect_keywords = {
        "service": ["staff", "waiter", "service", "horrible to us", "refilled"],
        "food": ["tasty", "food", "meat", "vegetable", "noodle", "pasta", "dish"],
        "price": ["overpriced", "expensive", "bill", "steep", "cheap", "cost"],
        "ambience": ["music", "candlelight", "atmosphere", "romantic", "trattoria"],
        "miscellaneous": ["parking", "subway", "table", "water sunken", "location"],
    }

    for aspect in target_aspects:
        keywords = aspect_keywords.get(aspect, [])
        detected[aspect] = any(kw in text_lower for kw in keywords)

    return detected


aspect_ontology = ["service", "food", "price", "ambience", "miscellaneous"]
for sample in aspect_data[:3]:
    eval_res = evaluate_aspects(sample["text"], aspect_ontology)
    active_labels = [k for k, v in eval_res.items() if v]
    text_val = sample["text"]
    print(f"\nText: {text_val!r}")
    print(f"Expected: {sample['labels']}")
    print(f"Detected: {active_labels}")
    for exp_label in sample["labels"]:
        assert exp_label in active_labels, f"Expected {exp_label} in {active_labels}"

print("\nMulti-label aspect classification tests passed successfully.")

## 3. Neural Machine Translation (NMT) Prompting Paradigms

Machine translation leverages in-context prompts to convert source language sequences $\mathbf{x} \in \mathcal{V}_{\text{src}}$ to target language sequences $\mathbf{y} \in \mathcal{V}_{\text{tgt}}$.

Empirical studies on LLM translation identify three prominent translation prompt patterns:
- **TP1 (Direct Directional):** `Translate these sentences from [SRC] to [TGT]:`
- **TP2 (Meaning Query):** `What do these sentences mean in [TGT]?`
- **TP3 (Polite Request):** `Please provide the [TGT] translation for these sentences:`

Let us evaluate these prompt patterns using our parallel translation corpus.


In [ ]:
with (DATA_DIR / "parallel_translation_corpus.json").open(encoding="utf-8") as f:
    translation_corpus = json.load(f)


def build_translation_prompt(
    src_text: str, src_lang: str, tgt_lang: str, template_type: str = "TP1"
) -> str:
    if template_type == "TP1":
        return f"Translate these sentences from {src_lang} to {tgt_lang}:\n\nText: {src_text}\nTranslation:"
    elif template_type == "TP2":
        return f"What do these sentences mean in {tgt_lang}?\n\nText: {src_text}\nMeaning:"
    elif template_type == "TP3":
        return f"Please provide the {tgt_lang} translation for these sentences:\n\nText: {src_text}\nTranslation:"
    else:
        raise ValueError(f"Unknown template type: {template_type}")


print("=== TRANSLATION TEMPLATE COMPARISON ===")
sample = translation_corpus[0]
for tp in ["TP1", "TP2", "TP3"]:
    prompt_str = build_translation_prompt(
        sample["source_text"], sample["source_lang"], sample["target_lang"], tp
    )
    print(f"--- {tp} ---")
    print(prompt_str)
    print()

# Verification assertions
assert len(translation_corpus) >= 4
assert sample["source_lang"] == "vi"
assert sample["target_lang"] == "en"
print("Machine translation templates verified.")